In [ ]:
from ts2.data.cell_data_module import CellDataModule 
import yaml
from omegaconf import OmegaConf
import torch
import matplotlib.pyplot as plt
import einops
import math
import torchvision
from PIL import Image
import numpy as np
from typing import List, Tuple
import time

In [ ]:
cf = """
infra:
  seed: 10
data:
  set: scsrh
  parser:
    which: CachedCSVParser
    params:
      cached_parser_file:
        train: ../data/tsmeta/srh7v1_sc/55e0deb2_cell_train
        trainval: ../data/tsmeta/srh7v1_sc/55e0deb2_cell_trainval
        test_databank: ../data/tsmeta/srh7v1_sc/c191d6d5_cell40_train
        test: ../data/tsmeta/srh7v1_sc/c191d6d5_cell40_trainval
  transform:
    train:
      which: HistologyTransform
      params:
        base_aug_params:
          laser_noise_config: null
          get_third_channel_params:
            mode: get_third_channel_cellmask_passthrough
            subtracted_base: 0.07629394531
          mask_params:
            how_to_process: rgba #small_patch
            bg_fill: [9957.5889, 10057.9795]
          to_uint8: True
        strong_aug_params:
          aug_list: []
          aug_prob: 1
    trainval: ${.train}
  dataset:
    which: CellDataset # SingleLevelHierarchicalDataset
    which_process_read: cell_memmap
    params:
      common:
        data_root: /nfs/turbo/umms-tocho-snr/exp/chengjia/scsrh_repl_root2/
        num_transforms: 1
        balance_instance_class: False
      train: {}
      trainval: {}
  loader:
    params:
      common:
        pin_memory: False
        persistent_workers: True
        prefetch_factor: 12
        num_workers: 7
      train:
        batch_size: 256
        drop_last: True
        shuffle: True
      trainval:
        batch_size: 64
        drop_last: False
        shuffle: False
"""

cf = OmegaConf.create(yaml.safe_load(cf))

In [ ]:
dm = CellDataModule(config=cf)
dm.setup(stage="fit")

In [ ]:
dset = dm.trainval_dataset_

In [ ]:
len(dset)

In [ ]:
k = 529
pit_idx = torch.where(torch.tensor([i["label"]=="pituita" for i in dset.instances_]))[0]
pit_sample_idx = sorted(pit_idx[torch.randperm(len(pit_idx))][:k])
pit_data = [dset.__getitem__(i) for i in pit_sample_idx]

In [ ]:
all_images = torch.stack([einops.rearrange(s["image"], "1 c h w -> h w c") for s in pit_data])
all_images_transparent = torch.clone(all_images)
all_images[...,-1] = 255
all_images_transparent[...,-1] = ((all_images_transparent[...,-1].to(torch.int32)+255)/2).to(torch.uint8)

In [ ]:
all_im_tile = einops.rearrange(
    torch.nn.functional.pad(
    all_images, pad=(0,0,5,5,5,5), value=255),
    "(bh bw) h w c -> (bh h) (bw w) c", bh=int(math.sqrt(k)), bw=int(math.sqrt(k)))
all_images_transparent = einops.rearrange(
    torch.nn.functional.pad(
    all_images_transparent, pad=(0,0,5,5,5,5), value=255),
    "(bh bw) h w c -> (bh h) (bw w) c", bh=int(math.sqrt(k)), bw=int(math.sqrt(k)))

In [ ]:
fig, ax=plt.subplots(2,1,figsize=(20,40))
ax[0].imshow(all_im_tile)
ax[1].imshow(all_images_transparent)

ax[0].axis("off")
ax[1].axis("off")